# cuBLAS Matmul Baseline on Jetson Orin Nano

Pure-Python (ctypes) wrapper around CUDA Driver API + cuBLAS.
No PyCUDA, no CuPy — just `libcuda.so` and `libcublas.so`.

Use this as a **performance baseline** for tiny-ton GEMM kernels.

In [12]:
import ctypes, ctypes.util, time
import numpy as np

# ── Load shared libraries ──────────────────────────────────────────

cuda = ctypes.cdll.LoadLibrary("libcuda.so")
cublas = ctypes.cdll.LoadLibrary("libcublas.so")
cudart = ctypes.cdll.LoadLibrary("libcudart.so")

# ── CUDA Driver API helpers ────────────────────────────────────────

def check_cuda(err, msg=""):
    if err != 0:
        raise RuntimeError(f"CUDA error {err}: {msg}")

def check_cublas(err, msg=""):
    if err != 0:
        raise RuntimeError(f"cuBLAS error {err}: {msg}")

check_cuda(cuda.cuInit(0), "cuInit")

device = ctypes.c_int()
check_cuda(cuda.cuDeviceGet(ctypes.byref(device), 0), "cuDeviceGet")

name_buf = ctypes.create_string_buffer(256)
cuda.cuDeviceGetName(name_buf, 256, device)

major, minor = ctypes.c_int(), ctypes.c_int()
cuda.cuDeviceGetAttribute(ctypes.byref(major), 75, device)  # CU_DEVICE_ATTRIBUTE_COMPUTE_CAPABILITY_MAJOR
cuda.cuDeviceGetAttribute(ctypes.byref(minor), 76, device)  # CU_DEVICE_ATTRIBUTE_COMPUTE_CAPABILITY_MINOR

print(f"Device: {name_buf.value.decode()}")
print(f"Compute capability: sm_{major.value}{minor.value}")

Device: Orin
Compute capability: sm_87


In [13]:
# ── cudaMalloc / cudaMemcpy / cudaFree via cudart ─────────────────

cudart.cudaMalloc.restype = ctypes.c_int
cudart.cudaMalloc.argtypes = [ctypes.POINTER(ctypes.c_void_p), ctypes.c_size_t]

cudart.cudaMemcpy.restype = ctypes.c_int
cudart.cudaMemcpy.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_size_t, ctypes.c_int]

cudart.cudaFree.restype = ctypes.c_int
cudart.cudaFree.argtypes = [ctypes.c_void_p]

cudart.cudaDeviceSynchronize.restype = ctypes.c_int

cudart.cudaEventCreate.restype = ctypes.c_int
cudart.cudaEventCreate.argtypes = [ctypes.POINTER(ctypes.c_void_p)]

cudart.cudaEventRecord.restype = ctypes.c_int
cudart.cudaEventRecord.argtypes = [ctypes.c_void_p, ctypes.c_void_p]

cudart.cudaEventSynchronize.restype = ctypes.c_int
cudart.cudaEventSynchronize.argtypes = [ctypes.c_void_p]

cudart.cudaEventElapsedTime.restype = ctypes.c_int
cudart.cudaEventElapsedTime.argtypes = [ctypes.POINTER(ctypes.c_float), ctypes.c_void_p, ctypes.c_void_p]

cudart.cudaEventDestroy.restype = ctypes.c_int
cudart.cudaEventDestroy.argtypes = [ctypes.c_void_p]

MEMCPY_H2D = 1
MEMCPY_D2H = 2

def gpu_alloc(nbytes):
    ptr = ctypes.c_void_p()
    check_cuda(cudart.cudaMalloc(ctypes.byref(ptr), nbytes), "cudaMalloc")
    return ptr

def gpu_copy_h2d(dst, src_np):
    check_cuda(cudart.cudaMemcpy(dst, src_np.ctypes.data, src_np.nbytes, MEMCPY_H2D), "H2D")

def gpu_copy_d2h(dst_np, src):
    check_cuda(cudart.cudaMemcpy(dst_np.ctypes.data, src, dst_np.nbytes, MEMCPY_D2H), "D2H")

def gpu_free(ptr):
    cudart.cudaFree(ptr)

print("CUDA runtime helpers ready.")

CUDA runtime helpers ready.


In [14]:
# ── cuBLAS handle ─────────────────────────────────────────────────

cublas.cublasCreate_v2.restype = ctypes.c_int
cublas.cublasCreate_v2.argtypes = [ctypes.POINTER(ctypes.c_void_p)]

cublas.cublasDestroy_v2.restype = ctypes.c_int
cublas.cublasDestroy_v2.argtypes = [ctypes.c_void_p]

cublas.cublasSgemm_v2.restype = ctypes.c_int
cublas.cublasSgemm_v2.argtypes = [
    ctypes.c_void_p,  # handle
    ctypes.c_int,     # transa
    ctypes.c_int,     # transb
    ctypes.c_int,     # m
    ctypes.c_int,     # n
    ctypes.c_int,     # k
    ctypes.POINTER(ctypes.c_float),  # alpha
    ctypes.c_void_p,  # A
    ctypes.c_int,     # lda
    ctypes.c_void_p,  # B
    ctypes.c_int,     # ldb
    ctypes.POINTER(ctypes.c_float),  # beta
    ctypes.c_void_p,  # C
    ctypes.c_int,     # ldc
]

cublas.cublasHgemm.restype = ctypes.c_int
cublas.cublasHgemm.argtypes = [
    ctypes.c_void_p,  # handle
    ctypes.c_int,     # transa
    ctypes.c_int,     # transb
    ctypes.c_int,     # m
    ctypes.c_int,     # n
    ctypes.c_int,     # k
    ctypes.c_void_p,  # alpha (pointer to __half)
    ctypes.c_void_p,  # A
    ctypes.c_int,     # lda
    ctypes.c_void_p,  # B
    ctypes.c_int,     # ldb
    ctypes.c_void_p,  # beta (pointer to __half)
    ctypes.c_void_p,  # C
    ctypes.c_int,     # ldc
]

handle = ctypes.c_void_p()
check_cublas(cublas.cublasCreate_v2(ctypes.byref(handle)), "cublasCreate")
print(f"cuBLAS handle: {handle.value:#x}")

cuBLAS handle: 0xaaab09a2a9f0


In [15]:
# ── Benchmark helper ──────────────────────────────────────────────

CUBLAS_OP_N = 0
CUBLAS_OP_T = 1

def cublas_sgemm(handle, A_dev, B_dev, C_dev, M, N, K):
    """C = A @ B using cuBLAS SGEMM (FP32).

    cuBLAS is column-major, so we compute C^T = B^T @ A^T
    which gives us row-major C = A @ B without transposing.
    """
    alpha = ctypes.c_float(1.0)
    beta = ctypes.c_float(0.0)
    check_cublas(cublas.cublasSgemm_v2(
        handle,
        CUBLAS_OP_N, CUBLAS_OP_N,
        N, M, K,
        ctypes.byref(alpha),
        B_dev, N,
        A_dev, K,
        ctypes.byref(beta),
        C_dev, N,
    ), "cublasSgemm")

def cublas_hgemm(handle, A_dev, B_dev, C_dev, M, N, K):
    """C = A @ B using cuBLAS HGEMM (FP16). Uses tensor cores on sm_87."""
    alpha = np.array([1.0], dtype=np.float16)
    beta = np.array([0.0], dtype=np.float16)
    check_cublas(cublas.cublasHgemm(
        handle,
        CUBLAS_OP_N, CUBLAS_OP_N,
        N, M, K,
        alpha.ctypes.data,
        B_dev, N,
        A_dev, K,
        beta.ctypes.data,
        C_dev, N,
    ), "cublasHgemm")

def benchmark_matmul(gemm_fn, A_np, B_np, dtype, warmup=5, repeats=20):
    """Allocate, copy, run, time, verify, report."""
    M, K = A_np.shape
    K2, N = B_np.shape
    assert K == K2

    A = A_np.astype(dtype)
    B = B_np.astype(dtype)
    C = np.zeros((M, N), dtype=dtype)

    A_dev = gpu_alloc(A.nbytes)
    B_dev = gpu_alloc(B.nbytes)
    C_dev = gpu_alloc(C.nbytes)
    gpu_copy_h2d(A_dev, A)
    gpu_copy_h2d(B_dev, B)

    # warmup
    for _ in range(warmup):
        gemm_fn(handle, A_dev, B_dev, C_dev, M, N, K)
    cudart.cudaDeviceSynchronize()

    # timed runs using CUDA events
    start_event = ctypes.c_void_p()
    end_event = ctypes.c_void_p()
    cudart.cudaEventCreate(ctypes.byref(start_event))
    cudart.cudaEventCreate(ctypes.byref(end_event))

    cudart.cudaEventRecord(start_event, None)
    for _ in range(repeats):
        gemm_fn(handle, A_dev, B_dev, C_dev, M, N, K)
    cudart.cudaEventRecord(end_event, None)
    cudart.cudaEventSynchronize(end_event)

    elapsed_ms = ctypes.c_float()
    cudart.cudaEventElapsedTime(ctypes.byref(elapsed_ms), start_event, end_event)
    avg_ms = elapsed_ms.value / repeats

    # correctness check
    gpu_copy_d2h(C, C_dev)
    expected = (A_np.astype(np.float32) @ B_np.astype(np.float32)).astype(dtype)
    max_err = float(np.abs(C.astype(np.float32) - expected.astype(np.float32)).max())

    # compute TFLOPs
    flops = 2.0 * M * N * K
    tflops = flops / (avg_ms * 1e-3) / 1e12

    gpu_free(A_dev)
    gpu_free(B_dev)
    gpu_free(C_dev)
    cudart.cudaEventDestroy(start_event)
    cudart.cudaEventDestroy(end_event)

    return avg_ms, tflops, max_err

print("Benchmark helper ready.")

Benchmark helper ready.


## FP32 SGEMM (no tensor cores)

In [16]:
np.random.seed(42)

print(f"{'Shape':>20s}  {'Time (ms)':>10s}  {'TFLOPS':>8s}  {'Max Err':>10s}")
print("-" * 56)

for size in [256, 512, 1024, 2048, 4096]:
    M = N = K = size
    A_np = np.random.randn(M, K).astype(np.float32)
    B_np = np.random.randn(K, N).astype(np.float32)

    avg_ms, tflops, max_err = benchmark_matmul(
        cublas_sgemm, A_np, B_np, np.float32)

    shape = f"{M}x{N}x{K}"
    print(f"{shape:>20s}  {avg_ms:10.3f}  {tflops:8.2f}  {max_err:10.4f}")

               Shape   Time (ms)    TFLOPS     Max Err
--------------------------------------------------------
         256x256x256       0.154      0.22      0.0000
         512x512x512       0.767      0.35      0.0001
      1024x1024x1024       2.450      0.88      0.0002
      2048x2048x2048      12.945      1.33      0.0003
      4096x4096x4096     102.088      1.35      0.0006


## FP16 HGEMM (uses tensor cores on sm_87)

In [17]:
np.random.seed(42)

print(f"{'Shape':>20s}  {'Time (ms)':>10s}  {'TFLOPS':>8s}  {'Max Err':>10s}")
print("-" * 56)

for size in [256, 512, 1024, 2048, 4096]:
    M = N = K = size
    A_np = np.random.randn(M, K).astype(np.float32)
    B_np = np.random.randn(K, N).astype(np.float32)

    avg_ms, tflops, max_err = benchmark_matmul(
        cublas_hgemm, A_np, B_np, np.float16)

    shape = f"{M}x{N}x{K}"
    print(f"{shape:>20s}  {avg_ms:10.3f}  {tflops:8.2f}  {max_err:10.4f}")

               Shape   Time (ms)    TFLOPS     Max Err
--------------------------------------------------------
         256x256x256       0.034      0.98      0.1250
         512x512x512       0.112      2.39      0.3125
      1024x1024x1024       0.629      3.41      0.6250
      2048x2048x2048       3.070      5.60      1.3750
      4096x4096x4096      12.588     10.92      3.0000


## NumPy CPU Baseline

In [18]:
np.random.seed(42)

print(f"{'Shape':>16s}  {'Time (ms)':>10s}  {'TFLOPS':>8s}")
print("-" * 40)

for size in [256, 512, 1024, 2048]:
    M = N = K = size
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)

    # warmup
    _ = A @ B

    t0 = time.perf_counter()
    repeats = 5
    for _ in range(repeats):
        C = A @ B
    elapsed = time.perf_counter() - t0
    avg_ms = (elapsed / repeats) * 1000

    flops = 2.0 * M * N * K
    tflops = flops / (avg_ms * 1e-3) / 1e12

    shape = f"{M}x{N}x{K}"
    print(f"{shape:>16s}  {avg_ms:10.3f}  {tflops:8.4f}")

           Shape   Time (ms)    TFLOPS
----------------------------------------
     256x256x256       0.898    0.0374
     512x512x512       2.037    0.1318
  1024x1024x1024      18.961    0.1133
  2048x2048x2048     144.389    0.1190


## Summary

In [19]:
np.random.seed(42)

print(f"{'Method':<20s}  {'Shape':>20s}  {'Time (ms)':>10s}  {'TFLOPS':>8s}")
print("-" * 64)

for SIZE in [2048, 4096]:
    M = N = K = SIZE
    A_np = np.random.randn(M, K).astype(np.float32)
    B_np = np.random.randn(K, N).astype(np.float32)
    shape = f"{M}x{N}x{K}"

    sgemm_ms, sgemm_tf, _ = benchmark_matmul(cublas_sgemm, A_np, B_np, np.float32)
    hgemm_ms, hgemm_tf, _ = benchmark_matmul(cublas_hgemm, A_np, B_np, np.float16)

    _ = A_np @ B_np
    t0 = time.perf_counter()
    for _ in range(5):
        _ = A_np @ B_np
    numpy_ms = (time.perf_counter() - t0) / 5 * 1000
    numpy_tf = 2.0 * M * N * K / (numpy_ms * 1e-3) / 1e12

    print(f"{'NumPy CPU':<20s}  {shape:>20s}  {numpy_ms:10.2f}  {numpy_tf:8.4f}")
    print(f"{'cuBLAS SGEMM':<20s}  {shape:>20s}  {sgemm_ms:10.2f}  {sgemm_tf:8.4f}")
    print(f"{'cuBLAS HGEMM':<20s}  {shape:>20s}  {hgemm_ms:10.2f}  {hgemm_tf:8.4f}")
    print()

print("HGEMM uses tensor cores (sm_87 Ampere).")
print("This is the cuBLAS target to beat with tiny-ton.")

Method                               Shape   Time (ms)    TFLOPS
----------------------------------------------------------------
NumPy CPU                   2048x2048x2048      137.56    0.1249
cuBLAS SGEMM                2048x2048x2048       13.00    1.3218
cuBLAS HGEMM                2048x2048x2048        2.84    6.0453

NumPy CPU                   4096x4096x4096      985.32    0.1395
cuBLAS SGEMM                4096x4096x4096      102.12    1.3459
cuBLAS HGEMM                4096x4096x4096       12.41   11.0752

HGEMM uses tensor cores (sm_87 Ampere).
This is the cuBLAS target to beat with tiny-ton.


## LLM-Shaped Matmuls (non-square)

Real LLM inference uses non-square matrices:
- **Single token:** M=1, N=4096, K=4096 (matvec)
- **Small batch:** M=32, N=4096, K=4096
- **MLP layer:** M=128, N=11008, K=4096 (Llama-style)
- **Attention QKV:** M=128, N=4096, K=4096

In [20]:
np.random.seed(42)

llm_shapes = [
    (1,    4096, 4096,  "single token (matvec)"),
    (8,    4096, 4096,  "8-token batch"),
    (32,   4096, 4096,  "32-token batch"),
    (128,  4096, 4096,  "128-token batch"),
    (512,  4096, 4096,  "512-token batch"),
    (128, 11008, 4096,  "MLP up-proj (Llama-7B)"),
    (128,  4096, 11008, "MLP down-proj (Llama-7B)"),
]

print(f"{'Description':<28s}  {'M':>5s} {'N':>5s} {'K':>5s}  {'FP32 ms':>8s} {'FP32 TF':>8s}  {'FP16 ms':>8s} {'FP16 TF':>8s}")
print("-" * 90)

for M, N, K, desc in llm_shapes:
    A_np = np.random.randn(M, K).astype(np.float32)
    B_np = np.random.randn(K, N).astype(np.float32)

    s_ms, s_tf, _ = benchmark_matmul(cublas_sgemm, A_np, B_np, np.float32)
    h_ms, h_tf, _ = benchmark_matmul(cublas_hgemm, A_np, B_np, np.float16)

    print(f"{desc:<28s}  {M:5d} {N:5d} {K:5d}  {s_ms:8.3f} {s_tf:8.2f}  {h_ms:8.3f} {h_tf:8.2f}")

Description                       M     N     K   FP32 ms  FP32 TF   FP16 ms  FP16 TF
------------------------------------------------------------------------------------------
single token (matvec)             1  4096  4096     1.680     0.02     1.106     0.03
8-token batch                     8  4096  4096     2.212     0.12     1.153     0.23
32-token batch                   32  4096  4096     2.275     0.47     1.130     0.95
128-token batch                 128  4096  4096     4.766     0.90     1.349     3.18
512-token batch                 512  4096  4096    12.860     1.34     2.455     7.00
MLP up-proj (Llama-7B)          128 11008  4096     8.286     1.39     2.022     5.71
MLP down-proj (Llama-7B)        128  4096 11008    10.455     1.10     2.660     4.34


## Cleanup

In [ ]:
cublas.cublasDestroy_v2(handle)
print("cuBLAS handle destroyed. Done.")

cuBLAS handle destroyed. Done.


: 